In [4]:
import spacy
from spacy.language import Language
import torch
from transformers import AutoTokenizer, AutoModelForTokenClassification
from spacy.tokens import Span

# -------------------------------------------------------
# 1) Load Czech transformer NER model
# -------------------------------------------------------

MODEL_NAME = "ufal/robeczech-base"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, add_prefix_space=True)
model = AutoModelForTokenClassification.from_pretrained(MODEL_NAME)
id2label = model.config.id2label


# -------------------------------------------------------
# 2) Register custom spaCy component
# -------------------------------------------------------

@Language.component("czech_ner")
def ner_component(doc):
    """Custom Czech NER component using a HF transformer model."""
    tokens = [t.text for t in doc]
    
    encoded = tokenizer(tokens, is_split_into_words=True, return_tensors="pt")

    with torch.no_grad():
        outputs = model(**encoded)

    pred_ids = outputs.logits.argmax(dim=-1).squeeze().tolist()

    entities = []
    current_start = None
    current_label = None

    # Skip CLS at index 0, iterate over doc tokens
    for i, token in enumerate(doc):
        pred = pred_ids[i + 1]  # offset by 1 because of CLS
        label = id2label[pred]

        if label.startswith("B-"):
            # close previous entity
            if current_start is not None:
                span = doc[current_start:i]
                span.label_ = current_label
                entities.append(span)

            current_start = i
            current_label = label[2:]

        elif label.startswith("I-") and current_label == label[2:]:
            # inside the same entity
            continue

        else:
            # end entity
            if current_start is not None:
                span = doc[current_start:i]
                span.label_ = current_label
                entities.append(span)
                current_start = None
                current_label = None

    # Final entity
    if current_start is not None:
        span = doc[current_start:len(doc)]
        span.label_ = current_label
        entities.append(span)

    doc.ents = entities
    return doc


# -------------------------------------------------------
# 3) Build spaCy pipeline and add component
# -------------------------------------------------------

nlp = spacy.blank("cs")
nlp.add_pipe("czech_ner")   # << works now
print("LOADED")


# -------------------------------------------------------
# 4) Run Czech NER
# -------------------------------------------------------

text = "Praha je hlavním městem České republiky. Tomáš Garrigue Masaryk byl prvním prezidentem."

doc = nlp(text)
print("Entities:")
print(doc)

for ent in doc.ents:
    print(ent.text, ent.label_)


Some weights of RobertaForTokenClassification were not initialized from the model checkpoint at ufal/robeczech-base and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


LOADED
Entities:
Praha je hlavním městem České republiky. Tomáš Garrigue Masaryk byl prvním prezidentem.


In [10]:
from transformers import AutoTokenizer, AutoModelForTokenClassification, pipeline

# Use the fine-tuned model name for the model weights
model_name = "popelucha/robeczech-NER"
# Use the BASE model name for the tokenizer (where the necessary files exist)
tokenizer_name = "ufal/robeczech-base"

# 1. Load the tokenizer from the BASE model
# The tokenizer is what's failing, so we load it from the complete repo
tokenizer = AutoTokenizer.from_pretrained(tokenizer_name)

# 2. Load the model weights from the fine-tuned repo
# This is where the actual NER knowledge is stored
model = AutoModelForTokenClassification.from_pretrained(model_name)

# 3. Create a NER pipeline for easy inference
ner_pipeline = pipeline("ner", model=model, tokenizer=tokenizer, aggregation_strategy="none")

# 4. Example usage (should now work)
text = "Prezident České republiky Petr Pavel navštívil Českou lípu."
results = ner_pipeline(text)
print(results)

Device set to use cuda:0


[{'entity': 'LABEL_0', 'score': np.float32(0.8849537), 'index': 1, 'word': 'Prezident', 'start': 0, 'end': 9}, {'entity': 'LABEL_13', 'score': np.float32(0.2337863), 'index': 2, 'word': 'ĠÄĮeskÃ©', 'start': 10, 'end': 15}, {'entity': 'LABEL_5', 'score': np.float32(0.2847991), 'index': 3, 'word': 'Ġrepubliky', 'start': 16, 'end': 25}, {'entity': 'LABEL_9', 'score': np.float32(0.43741417), 'index': 4, 'word': 'ĠPetr', 'start': 26, 'end': 30}, {'entity': 'LABEL_2', 'score': np.float32(0.46545455), 'index': 5, 'word': 'ĠPavel', 'start': 31, 'end': 36}, {'entity': 'LABEL_0', 'score': np.float32(0.9096907), 'index': 6, 'word': 'ĠnavÅ¡tÃŃvil', 'start': 37, 'end': 46}, {'entity': 'LABEL_13', 'score': np.float32(0.35209286), 'index': 7, 'word': 'ĠÄĮeskou', 'start': 47, 'end': 53}, {'entity': 'LABEL_5', 'score': np.float32(0.2492106), 'index': 8, 'word': 'ĠlÃŃ', 'start': 54, 'end': 56}, {'entity': 'LABEL_5', 'score': np.float32(0.25850102), 'index': 9, 'word': 'pu', 'start': 56, 'end': 58}, {'en

In [ ]:
import requests

url = "http://localhost:7200/repositories/nkod-test-repo/statements" #"http://LAPTOP-F4189N2L:7200/repositories/nkod-repository/statements"#
file_path = "./four-thousanders.ttl"

with open(file_path, "rb") as f:
    headers = {
        "Content-Type": "application/x-turtle"
    }
    response = requests.get(url)#requests.post(url, headers=headers, data=f)

print("Status:", response.status_code)
print("Response:", response.text)


In [ ]:
import requests

url = "http://localhost:7200/repositories/nkod-test-repo/statements"
params = {"context": "<http://example.org/fourGraph>"}
headers = {"Content-Type": "text/turtle"}

with open("./four-thousanders.ttl", "rb") as f:
    r = requests.post(url, params=params, headers=headers, data=f)

print(r.status_code, r.text)


204 


In [ ]:
import requests

file_path = "./four-thousanders.ttl"
graphdb_repo = "nkod-test-repo"
named_graph_uri = "<http://example.org/fourGraph>"

with open(file_path, 'rb') as f:
    file_content = f.read()
headers = {'Content-Type': 'application/x-turtle', 'Accept': 'application/json'}

upload_url = f'http://localhost:7200/repositories/{graphdb_repo}/rdf-graphs/service?graph={named_graph_uri}'
requests.post(upload_url, headers=headers, data=file_content)

<Response [400]>

In [ ]:
import requests

REPO_URL = "http://localhost:7200/repositories/nkod-test-repo"

query = """
SELECT DISTINCT ?g
WHERE {
  GRAPH ?g {}
}
"""

headers = {
    "Content-Type": "application/sparql-query",
    "Accept": "application/sparql-results+json"
}

response = requests.post(REPO_URL, data=query, headers=headers)

data = response.json()

graphs = [binding for binding in data["results"]["bindings"]]
print(graphs)

print("Named graphs:")
for g in graphs:
    print(" -", g)


[{}]
Named graphs:
 - {}


In [ ]:
import requests

url = "http://sparql.xxxxxx.com/repositories/nkod_repository/statements"
file_path = "./four-thousanders.ttl"

with open(file_path, "rb") as f:
    headers = {
        "Content-Type": "application/x-turtle"
    }
    response = requests.post(url, headers=headers, data=f)

print("Status:", response.status_code)
print("Response:", response.text)


Status: 522
Response: <!DOCTYPE html>
<!--[if lt IE 7]> <html class="no-js ie6 oldie" lang="en-US"> <![endif]-->
<!--[if IE 7]>    <html class="no-js ie7 oldie" lang="en-US"> <![endif]-->
<!--[if IE 8]>    <html class="no-js ie8 oldie" lang="en-US"> <![endif]-->
<!--[if gt IE 8]><!--> <html class="no-js" lang="en-US"> <!--<![endif]-->
<head>

<title>xxxxx.com | 522: Connection timed out</title>
<meta charset="UTF-8" />
<meta http-equiv="Content-Type" content="text/html; charset=UTF-8" />
<meta http-equiv="X-UA-Compatible" content="IE=Edge" />
<meta name="robots" content="noindex, nofollow" />
<meta name="viewport" content="width=device-width,initial-scale=1" />
<link rel="stylesheet" id="cf_styles-css" href="/cdn-cgi/styles/main.css" />
</head>
<body>
<div id="cf-wrapper">
    <div id="cf-error-details" class="p-0">
        <header class="mx-auto pt-10 lg:pt-6 lg:px-8 w-240 lg:w-full mb-8">
            <h1 class="inline-block sm:block sm:mb-2 font-light text-60 lg:text-4xl text-black-d

In [12]:
import os
import requests
from dotenv import load_dotenv


load_dotenv()

GRAPHDB_URL = os.getenv("GRAPHDB_URL", "http://osw.felk.cvut.cz:7200/repositories/lamossta")
GRAPHDB_USERNAME = os.getenv("GRAPHDB_USERNAME", "lamossta")
GRAPHDB_PASSWORD = os.getenv("GRAPHDB_PASSWORD", "S$i#9KoYNxA!cQ")
NAMED_GRAPH = "http%3A%2F%2Fexample.org%2Ffour-thousanders2"
RDF_FILE_PATH = "./input_2025-11-25_02-44-45.jsonld"

def load_data_to_graphdb():
    print(GRAPHDB_URL, GRAPHDB_USERNAME, GRAPHDB_PASSWORD)
    api_url = f"{GRAPHDB_URL}/rdf-graphs/service?graph={NAMED_GRAPH}"
    
    with open(RDF_FILE_PATH, 'rb') as f:
        data = f.read()
    

    headers = {
        "Content-Type": "application/ld+json" 
    }
    
    print(f"Attempting to load data into graph: {NAMED_GRAPH} at {api_url}")
    
    try:
        response = requests.post(
            api_url,
            data=data,
            headers=headers,
            auth=(GRAPHDB_USERNAME, GRAPHDB_PASSWORD) # Basic Authentication
        )

        if response.status_code == 204:
            # 204 No Content is the standard success code for this operation
            print("✅ Success: Data loaded successfully into the named graph!")
        else:
            print(f"❌ API Error: Status code {response.status_code}")
            print(f"Response text: {response.text}")
            
    except requests.exceptions.RequestException as e:
        print(f"Fatal connection error: {e}")

if __name__ == "__main__":
    load_data_to_graphdb()

http://osw.felk.cvut.cz:7200/repositories/lamossta lamossta S$i#9KoYNxA!cQ
Attempting to load data into graph: http%3A%2F%2Fexample.org%2Ffour-thousanders2 at http://osw.felk.cvut.cz:7200/repositories/lamossta/rdf-graphs/service?graph=http%3A%2F%2Fexample.org%2Ffour-thousanders2
✅ Success: Data loaded successfully into the named graph!


In [6]:
import os
import requests
from dotenv import load_dotenv
import json

load_dotenv()

GRAPHDB_URL = os.getenv("GRAPHDB_URL", "http://osw.felk.cvut.cz:7200/repositories/lamossta")
GRAPHDB_USERNAME = os.getenv("GRAPHDB_USERNAME", "lamossta")
GRAPHDB_PASSWORD = os.getenv("GRAPHDB_PASSWORD", "S$i#9KoYNxA!cQ")
NAMED_GRAPH = "http://example.org/four-thousanders1"

def sparql_query_graphdb(query: str):
    api_url = GRAPHDB_URL  # <-- IMPORTANT: not /statements for SELECT queries
    
    headers = {
        "Accept": "application/sparql-results+json",
        "Content-Type": "application/sparql-query",
    }
    
    print("Sending raw SPARQL query:\n", query)

    try:
        response = requests.post(
            api_url,
            data=query,   # <-- RAW QUERY, NOT FORM ENCODED
            headers=headers,
            auth=(GRAPHDB_USERNAME, GRAPHDB_PASSWORD)
        )

        print("Status:", response.status_code)

        if response.status_code == 200:
            print("Response:")
            print(response.text)
            return response.text

        print("Error response:")
        print(response.text)
        return None

    except requests.exceptions.RequestException as e:
        print("Fatal error:", e)
        return None


if __name__ == "__main__":
    SAMPLE_SPARQL_QUERY = """
    SELECT ?s ?p ?o
    FROM <http://example.org/four-thousanders1>
    WHERE {
      ?s ?p ?o .
    }
    LIMIT 10
    """
    
    sparql_query_graphdb(SAMPLE_SPARQL_QUERY)


Sending raw SPARQL query:
 
    SELECT ?s ?p ?o
    FROM <http://example.org/four-thousanders1>
    WHERE {
      ?s ?p ?o .
    }
    LIMIT 10
    
Status: 200
Response:
{
  "head" : {
    "vars" : [
      "s",
      "p",
      "o"
    ]
  },
  "results" : {
    "bindings" : [
      {
        "s" : {
          "type" : "uri",
          "value" : "https://www.theuiaa.org/4000-alps/summit/Mont-Blanc"
        },
        "p" : {
          "type" : "uri",
          "value" : "http://www.w3.org/1999/02/22-rdf-syntax-ns#type"
        },
        "o" : {
          "type" : "uri",
          "value" : "https://www.theuiaa.org/4000-alps/schema/summit"
        }
      },
      {
        "s" : {
          "type" : "uri",
          "value" : "https://www.theuiaa.org/4000-alps/summit/Mont-Blanc-de-Courmayeur"
        },
        "p" : {
          "type" : "uri",
          "value" : "http://www.w3.org/1999/02/22-rdf-syntax-ns#type"
        },
        "o" : {
          "type" : "uri",
          "value" 

In [ ]:
import json
from rdflib import Dataset
import rdflib
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

rdflib.plugins.sparql.SPARQL_LOAD_GRAPHS = False


import json
from rdflib import Dataset, Graph, URIRef
from concurrent.futures import ThreadPoolExecutor

# -----------------------------
# Step 1: Sample JSON-LD data
# -----------------------------
jsonld_data = [
    {
        "@context": {"name": "http://schema.org/name", "age": "http://schema.org/age"},
        "@id": "http://example.org/person1",
        "name": "Alice",
        "age": 30
    },
    {
        "@context": {"name": "http://schema.org/name", "age": "http://schema.org/age"},
        "@id": "http://example.org/person2",
        "name": "Bob",
        "age": 25
    },
    {
        "@context": {"name": "http://schema.org/name", "age": "http://schema.org/age"},
        "@id": "http://example.org/person3",
        "name": "Charlie",
        "age": 35
    }
]

# -----------------------------
# Step 2: Create Dataset and add Graphs
# -----------------------------
dataset = Dataset()
GRAPH_IRI = "http://example.org/graph"

for idx, data in enumerate(jsonld_data):
    # Create a unique identifier for each graph
    identifier = URIRef(f"{GRAPH_IRI}/g{idx+1}")
    g = Graph(identifier=identifier)

    # Convert JSON-LD dict to string
    jsonld_str = json.dumps(data)

    # Parse JSON-LD into the graph
    g.parse("input_2025-11-25_02-44-45.jsonld", format="json-ld")

    # Add the graph to the dataset
    dataset.add_graph(g)

# -----------------------------
# Step 3: Define SPARQL query (use FROM <graph_uri>)
# -----------------------------
def query_graph(graph_uri):
    query = f"""
    PREFIX schema: <http://schema.org/>
    SELECT ?person ?name ?age

    FROM <{graph_uri}>
    FROM <http://example.org/graph>

    WHERE {{
        ?person ?name ?age .
    }}
    """
    return list(dataset.query(query))

# -----------------------------
# Step 4: Query all graphs in parallel
# -----------------------------
graph_uris = [str(URIRef(f"{GRAPH_IRI}/g{i+1}")) for i in range(len(jsonld_data))]

with ThreadPoolExecutor(max_workers=3) as executor:
    futures = [executor.submit(query_graph, uri) for uri in graph_uris]
    results = [f.result() for f in futures]

# -----------------------------
# Step 5: Print results
# -----------------------------
for i, res in enumerate(results, start=1):
    print(f"Results from graph {graph_uris[i-1]}:")
    for row in res:
        print(f"  Person: {row.person}, Name: {row.name}, Age: {row.age}")
    print()




Results from graph http://example.org/graph/g1:
  Person: https://vyzlovka.cz/4w-opendata.php?agenda=turisticke-cile&id=3, Name: http://purl.org/dc/terms/title, Age: Zaniklá osada Ve Spáleném
  Person: Nadab028fadd040fe972bda7b74d6d686, Name: http://purl.org/goodrelations/v1#hasPriceSpecification, Age: N3e04af3d72ff4daca8736b711b9c254d
  Person: https://vyzlovka.cz/4w-opendata.php?agenda=turisticke-cile&id=6, Name: http://purl.org/dc/terms/description, Age: Za Vyžlovským rybníkem pár metrů po žluté, na rozcestí pravou širokou lesní cestou (naučná stezka).
  Person: https://vyzlovka.cz/4w-opendata.php?agenda=turisticke-cile&id=3, Name: http://www.w3.org/1999/02/22-rdf-syntax-ns#type, Age: http://purl.org/dc/terms/Location
  Person: https://vyzlovka.cz/4w-opendata.php?agenda=turisticke-cile&id=5, Name: http://www.w3.org/1999/02/22-rdf-syntax-ns#type, Age: http://schema.org/TouristAttraction
  Person: https://vyzlovka.cz/4w-opendata.php?agenda=turisticke-cile&id=5, Name: http://purl.org/d

In [14]:
import rdflib

rdflib.Graph().parse("input_2025-11-25_02-44-45.jsonld", format="json-ld")

<Graph identifier=N9bc8188ef0a849e6ac1f7ad8ac950f80 (<class 'rdflib.graph.Graph'>)>

In [7]:
import os
import requests
from dotenv import load_dotenv

def get_named_graphs(repository_id: str = "lamossta", graphdb_host: str = "http://osw.felk.cvut.cz:7200/repositories/lamossta") -> list:
    api_url = f"{graphdb_host}/repositories/{repository_id}/statements"
    
    headers = {
        'Accept': 'application/json'
    }
    
    # Optional: If your repository requires authentication for this call:
    username = os.getenv("GRAPHDB_USERNAME", "lamossta")
    password = os.getenv("GRAPHDB_PASSWORD", "S$i#9KoYNxA!cQ")
    auth_tuple = (username, password) if username and password else None
    
    print(f"Querying contexts from: {api_url}")
    
    try:
        # 3. Send the GET request
        response = requests.get(
            api_url,
            headers=headers,
            auth=auth_tuple  # Uncomment if authentication is needed
        )

        # 4. Check the response status
        if response.status_code == 200:
            # 5. Parse the JSON response
            data = response.json()
            
            # The structure is standard SPARQL JSON Results Format
            bindings = data.get('results', {}).get('bindings', [])
            
            # Extract the graph IRI (the 'contextID' variable)
            graph_iris = [item['contextID']['value'] for item in bindings]
            
            return graph_iris
        else:
            print(f"❌ API Error: Status code {response.status_code}")
            print(f"Response text: {response.text}")
            return []

    except requests.exceptions.RequestException as e:
        print(f"Fatal connection error: {e}")
        return []

get_named_graphs()

Querying contexts from: http://osw.felk.cvut.cz:7200/repositories/lamossta/repositories/lamossta/statements
❌ API Error: Status code 406
Response text: Not Acceptable



[]

In [1]:
import os

def process_file_paths(file_path):
    """
    Loads a text file, extracts the directory path (substring before the last '/') 
    from each line, and returns a list of these paths.

    Args:
        file_path (str): The path to the input text file.

    Returns:
        list: A list of strings, where each string is the directory path 
              extracted from a line in the file.
    """
    extracted_paths = []
    
    try:
        with open(file_path, 'r') as f:
            for line in f:
                # 1. Clean up the line (remove leading/trailing whitespace and newlines)
                clean_line = line.strip()
                
                # 2. Ensure the line is not empty
                if clean_line:
                    # 3. Find the last occurrence of '/' and get the substring before it
                    # os.path.dirname() is the standard way to get the directory part of a path
                    directory_path = os.path.dirname(clean_line)
                    print(directory_path.split('/')[-1])
                    
                    # 4. Do something with the extracted path - here, we append it to the list
                    extracted_paths.append(directory_path)
                    
    except FileNotFoundError:
        print(f"Error: The file '{file_path}' was not found.")
        return [] # Return an empty list on error
        
    return extracted_paths

# --- Example Usage ---

# 1. Create a temporary file with your data for demonstration
example_data = """
data/nkod/distributions/999788533/distribution.jsonld
data/nkod/distributions/994939632/distribution.jsonld 
data/nkod/distributions/994059770/distribution.jsonld
data/nkod/distributions/9234de229f146e408f0982a1877d003e/distribution.jsonld
"""
temp_file_name = "paths_to_process.txt"

with open(temp_file_name, 'w') as f:
    f.write(example_data.strip())

# 2. Call the function
result_paths = process_file_paths(temp_file_name)

# 3. Print the results
print("--- Extracted Directory Paths ---")
for path in result_paths:
    print(path)

# 4. Optional: Clean up the temporary file
# os.remove(temp_file_name)

999788533
994939632
994059770
9234de229f146e408f0982a1877d003e
--- Extracted Directory Paths ---
data/nkod/distributions/999788533
data/nkod/distributions/994939632
data/nkod/distributions/994059770
data/nkod/distributions/9234de229f146e408f0982a1877d003e
